In [ ]:
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import math
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from ipywidgets import interact
import ipywidgets as widgets

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Regularization
## Controlling Complexity, Generalization, and Invariance

This lecture explores regularization techniques in deep neural networks:
- **Overfitting & Generalization**: The trade-off between model capacity and test error
- **Weight Decay ($L_2$ Regularization)**: Eigenvalue shrinkage and Hessian interaction
- **$L_1$ Regularization & Sparsity**: Geometry of the $L_1$ diamond vs $L_2$ sphere
- **Early Stopping**: Implicit regularization and formal equivalence to weight decay
- **Dropout**: Model averaging through ensembles of thinned subnetworks (Srivastava et al. 2014)
- **Batch Normalization**: Stabilizing internal covariate shifts (Ioffe & Szegedy 2015)
- **Data Augmentation**: Incorporating domain transformation invariances

# Overfitting and Generalization
## Beyond Training Loss Minimization

Deep neural networks are typically **overparameterized**: the number of trainable weights $W$ often vastly exceeds the number of training samples $N$.

By the Universal Approximation Theorem, an unconstrained overparameterized network can memorize random labels without learning underlying generalizable patterns.

**Regularization** refers to any modification applied to a learning algorithm that is intended to reduce its generalization error without significantly increasing its training error:
- Explicit penalties added to the loss function (Weight decay, $L_1$ norm)
- Algorithmic constraints on the optimization trajectory (Early stopping)
- Stochastic perturbations during training (Dropout, Data augmentation)
- Architectural reparameterizations (Batch normalization, Weight normalization)

# Weight Decay ($L_2$ Regularization)
## Parameter Shrinkage

We augment the objective function with an $L_2$ penalty on the weights:
$$\tilde{E}(\mathbf{w}) = E(\mathbf{w}) + \frac{\lambda}{2} \|\mathbf{w}\|^2 = E(\mathbf{w}) + \frac{\lambda}{2} \mathbf{w}^\mathrm{T} \mathbf{w}$$

The gradient update under stochastic gradient descent becomes:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \eta \left(\nabla E(\mathbf{w}^{(\tau)}) + \lambda \mathbf{w}^{(\tau)}\right) = (1 - \eta \lambda) \mathbf{w}^{(\tau)} - \eta \nabla E(\mathbf{w}^{(\tau)})$$

The factor $(1 - \eta \lambda) < 1$ actively shrinks the weight vector toward zero at every iteration, giving rise to the name **weight decay**.

Near a minimum where the local quadratic approximation $E(\mathbf{w}) \approx E(\mathbf{w}^*) + \frac{1}{2}(\mathbf{w} - \mathbf{w}^*)^\mathrm{T} \mathbf{H} (\mathbf{w} - \mathbf{w}^*)$ holds:
$$\mathbf{w}_\mathrm{reg} = \sum_{i=1}^W \frac{\lambda_i}{\lambda_i + \lambda} (\mathbf{w}^*)^\mathrm{T} \mathbf{u}_i \mathbf{u}_i$$
Directions with high curvature ($\lambda_i \gg \lambda$) are preserved; directions with low curvature ($\lambda_i \ll \lambda$) are suppressed toward zero!

## $L_1$ Regularization: Inducing Sparsity

Alternatively, we add an $L_1$ norm penalty (Lasso):
$$\tilde{E}(\mathbf{w}) = E(\mathbf{w}) + \lambda \sum_{i=1}^W |w_i|$$

The subgradient step is:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \eta \nabla E(\mathbf{w}) - \eta \lambda \mathrm{sign}(\mathbf{w})$$

```
          L_2 Constraint: Circle                  L_1 Constraint: Diamond
                  ^ w_2                                   ^ w_2
                  |                                       |   /\
               .-'|'-.                                    |  /  \
             .'   |   '.                                  | /    \
          ---+----+----+---> w_1                       ---+------+---> w_1
             '.   |   .'                                  | \    /
               '-.|.-'                                    |  \  /
                  |                                       |   \/
```

> 📘 **Geometric Sparsity:** The corners of the $L_1$ diamond lie on the coordinate axes. The elliptical contours of the loss function intersect the constraint region at these corners, driving non-essential weights strictly to **exact zeros**.

# Early Stopping as Regularization
## Bounding Effective Parameter Norms

During iterative optimization, training error monotonically decreases, whereas validation error on unseen data typically decreases initially, reaches a minimum, and then rises due to overfitting.

**Early Stopping** terminates training at the epoch $\tau^*$ where validation loss reaches its minimum.

> 💡 **Formal Equivalence to Weight Decay:** For a quadratic error function initialized at $\mathbf{w}^{(0)} = \mathbf{0}$ and updated with learning rate $\eta$ for $\tau$ steps:

$$\mathbf{w}^{(\tau)} = \left[\mathbf{I} - (\mathbf{I} - \eta \mathbf{H})^\tau\right] \mathbf{w}^*$$

Comparing this with $L_2$ weight decay $\mathbf{w}_\mathrm{reg} = (\mathbf{H} + \lambda \mathbf{I})^{-1}\mathbf{H}\mathbf{w}^*$, the effective regularization strength satisfies:

$$\tau \eta \approx \frac{1}{\lambda}$$

Training for fewer iterations $\tau$ is mathematically equivalent to applying stronger $L_2$ weight penalty!

# Dropout
## Ensembles of Thinned Subnetworks

Introduced by Srivastava, Hinton, Krizhevsky, Sutskever, & Salakhutdinov (2014), **Dropout** prevents complex co-adaptations between hidden units by randomly deactivating neurons during training.

At each training iteration and for each sample:
$$\tilde{\mathbf{z}}^{(l)} = \mathbf{r}^{(l)} \odot \mathbf{z}^{(l)}, \quad r_j^{(l)} \sim \mathrm{Bernoulli}(1 - p)$$
where $p \in (0, 1)$ is the dropout probability (typically $p = 0.5$ for hidden layers).

<small>📖 *Source: [Bishop, C. M., & Bishop, H. (2024). Deep Learning: Foundations and Concepts. Springer (Section 9.5)](https://doi.org/10.1007/978-3-031-45468-4)*</small>

- **Training Phase**: Each forward step trains a different thinned network sampled from $2^M$ possible subnetworks sharing parameters.
- **Inverted Dropout**: During training, activations are scaled by $\frac{1}{1 - p}$ so that **no scaling is required at test time**:
  $$\tilde{z}_j = \frac{r_j}{1 - p} z_j$$
  At test time, the full network runs deterministically, approximating the geometric mean of the ensemble.

# Batch Normalization
## Stabilizing Internal Covariate Shifts

In deep networks, parameter updates in early layers alter the distribution of inputs received by deeper layers at every step.

**Batch Normalization** (Ioffe & Szegedy 2015) normalizes the activations of each mini-batch $\mathcal{B} = \{a_1, \dots, a_B\}$:

1. **Compute Mini-Batch Statistics**:
   $$\mu_\mathcal{B} = \frac{1}{B} \sum_{b=1}^B a_b, \quad \sigma_\mathcal{B}^2 = \frac{1}{B} \sum_{b=1}^B (a_b - \mu_\mathcal{B})^2$$
2. **Normalize to Zero Mean & Unit Variance**:
   $$\hat{a}_b = \frac{a_b - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^2 + \epsilon}}$$
3. **Scale and Shift (Learnable Parameters)**:
   $$y_b = \gamma \hat{a}_b + \beta$$

Where $\gamma$ and $\beta$ allow the network to recover the optimal activation distribution. During inference, running exponential moving averages $(\mu_\mathrm{run}, \sigma_\mathrm{run}^2)$ are used for deterministic evaluation.

In [ ]:
# Comparative Regularization Benchmark on FashionMNIST
class RegularizedMLP(nn.Module):
    def __init__(self, use_dropout=False, use_batchnorm=False, p_drop=0.5):
        super().__init__()
        layers = [nn.Linear(784, 128)]
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(128))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(p_drop))
            
        layers.append(nn.Linear(128, 64))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(64))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(p_drop))
            
        layers.append(nn.Linear(64, 10))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x.view(x.shape[0], -1))

# Instantiate variants
model_baseline = RegularizedMLP(use_dropout=False, use_batchnorm=False)
model_dropout = RegularizedMLP(use_dropout=True, p_drop=0.3)
model_bn = RegularizedMLP(use_batchnorm=True)

print("Instantiated Baseline, Dropout, and BatchNorm Architectures.")

In [ ]:
# Training step verification across regularized models
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST(root=hp.get_data_path(""), train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)

models = {'Baseline': model_baseline, 'Dropout (p=0.3)': model_dropout, 'BatchNorm': model_bn}
crit = nn.CrossEntropyLoss()

for name, m in models.items():
    m.train()
    opt = torch.optim.Adam(m.parameters(), lr=0.002, weight_decay=1e-4 if name == 'Baseline' else 0.0)
    for i, (imgs, lbls) in enumerate(train_loader):
        opt.zero_grad()
        loss = crit(m(imgs), lbls)
        loss.backward()
        opt.step()
        if i >= 30: # 30 mini-batches for fast interactive execution
            break
    print(f"Model: {name:18s} | 30-step Train Cross-Entropy Loss: {loss.item():.4f}")

In [ ]:
# Simulation of polynomial overfitting vs Ridge regularization lambda
def plot_poly_reg(degree=8, reg_lambda=0.0):
    torch.manual_seed(42)
    x_train = torch.linspace(0, 1, 15)
    t_train = torch.sin(2 * math.pi * x_train) + torch.randn(15) * 0.2
    
    x_dense = torch.linspace(0, 1, 200)
    t_true = torch.sin(2 * math.pi * x_dense)
    
    Phi_tr = torch.stack([x_train**p for p in range(degree + 1)], dim=1)
    Phi_dense = torch.stack([x_dense**p for p in range(degree + 1)], dim=1)
    
    reg_mat = reg_lambda * torch.eye(degree + 1)
    w = torch.linalg.solve(Phi_tr.T @ Phi_tr + reg_mat, Phi_tr.T @ t_train)
    y_pred = Phi_dense @ w
    
    plt.figure(figsize=(7, 3.8))
    plt.plot(x_dense.numpy(), t_true.numpy(), 'g--', lw=2, label='True sin(2pi x)')
    plt.plot(x_dense.numpy(), y_pred.numpy(), 'r-', lw=2.2, label=f'Model (deg={degree}, lambda={reg_lambda:.1e})')
    plt.scatter(x_train.numpy(), t_train.numpy(), color='blue', s=40, zorder=5, label='Noisy Data (N=15)')
    plt.ylim(-1.6, 1.6); plt.legend(loc='lower left'); plt.grid(True); plt.tight_layout(); plt.show()

In [ ]:
# Interactive exploration: Eliminating polynomial overfitting via weight decay
interact(plot_poly_reg, 
         degree=widgets.IntSlider(min=1, max=10, step=1, value=8),
         reg_lambda=widgets.FloatLogSlider(value=1e-4, base=10, min=-7, max=0, step=0.5));

# Summary: Regularization in Deep Learning (Part 1)

In this lecture, we examined modern regularization techniques:

1. **Parameter Penalties**:
   - $L_2$ Weight Decay shrinks weights along directions of low Hessian curvature, preventing overfitting.
   - $L_1$ Regularization produces sparse models by driving uninformative parameters strictly to zero.

2. **Early Stopping**:
   - Restricting gradient descent iterations $\tau^*$ is formally equivalent to bounding the effective weight norm ($\tau \eta \sim 1/\lambda$).

## Summary: Regularization in Deep Learning (Part 2)

3. **Dropout**:
   - Randomly deactivating units with probability $p$ forces nodes to learn robust, non-coadapted representations.
   - Inverted dropout enables seamless deterministic inference at test time without ensemble overhead.

4. **Batch Normalization**:
   - Normalizing intermediate activations to zero mean and unit variance stabilizes gradient backpropagation, enabling faster convergence and higher learning rates.